# Project 3 - Credit Risk Modeling (Academically Strong + Simple)

This notebook is designed to be both:
- technically solid,
- easy to explain in presentation.

It strictly follows the required scope:
- class imbalance handling,
- 4 mandatory models (Logistic, RF, XGBoost, MLP),
- AUC, F1, Confusion Matrix.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
# Optional dependencies checks
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
except Exception as exc:
    raise RuntimeError("Install imbalanced-learn: pip install -r requirements.txt") from exc

try:
    from xgboost import XGBClassifier
except Exception as exc:
    raise RuntimeError(
        "XGBoost unavailable. On Mac, run: brew install libomp"
    ) from exc


In [ ]:
# Configuration
OUTPUT_DIR = Path('outputs')
CM_DIR = OUTPUT_DIR / 'confusion_matrices'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CM_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
DEFAULT_THRESHOLD = 0.50

# Speed settings for section 7 (quick CV)
CV_FOLDS = 2
CV_SAMPLE_MAX = 12000
CV_INCLUDE_MLP = False

# Data loading: Colab upload or local fallback
try:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise ValueError("No uploaded file. Please run files.upload() again.")
    uploaded_filename = next(iter(uploaded.keys()))
    DATA_PATH = Path(uploaded_filename)
    print(f"Data loaded via Colab upload: {DATA_PATH}")
except ImportError:
    DATA_PATH = Path('data/default_of_credit_card_clients.xls')
    print(f"google.colab not found, local fallback path: {DATA_PATH}")


In [ ]:
def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace(".", "_")
        .replace("/", "_")
        .replace("-", "_")
    )


def load_credit_dataset(data_path: Path) -> pd.DataFrame:
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}")

    if data_path.suffix.lower() == ".csv":
        df = pd.read_csv(data_path)
    elif data_path.suffix.lower() in {".xls", ".xlsx"}:
        try:
            df = pd.read_excel(data_path)
            if "default.payment.next.month" not in [str(c) for c in df.columns]:
                df = pd.read_excel(data_path, header=1)
        except Exception:
            df = pd.read_excel(data_path, header=1)
    else:
        raise ValueError("Unsupported format. Use .csv/.xls/.xlsx")

    df = df.loc[:, ~df.columns.astype(str).str.contains("^Unnamed")]
    df.columns = [normalize_column_name(c) for c in df.columns]
    return df


def detect_target_column(columns: list[str]) -> str:
    candidates = [
        "default_payment_next_month",
        "default",
        "y",
        "target",
    ]
    for c in candidates:
        if c in columns:
            return c
    raise ValueError("Target column not found. Expected default.payment.next.month")


def add_feature_engineering(df: pd.DataFrame):
    df = df.copy()
    created = []

    # Utilization ratios
    for i in range(1, 7):
        bill = f"bill_amt{i}"
        if bill in df.columns and "limit_bal" in df.columns:
            col = f"util_ratio{i}"
            df[col] = df[bill] / (df["limit_bal"].replace(0, np.nan))
            df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
            created.append(col)

    # Payment / bill ratios
    for i in range(1, 7):
        pay = f"pay_amt{i}"
        bill = f"bill_amt{i}"
        if pay in df.columns and bill in df.columns:
            col = f"pay_to_bill_ratio{i}"
            df[col] = df[pay] / (df[bill].abs() + 1)
            df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
            created.append(col)

    # Delinquency metrics
    pay_status_cols = [
        c for c in df.columns
        if c.startswith("pay_") and not c.startswith("pay_amt")
    ]
    if len(pay_status_cols) > 0:
        df["delinquency_count"] = (df[pay_status_cols] > 0).sum(axis=1)
        df["delinquency_max"] = df[pay_status_cols].max(axis=1)
        df["delinquency_mean"] = df[pay_status_cols].mean(axis=1)
        created += ["delinquency_count", "delinquency_max", "delinquency_mean"]

    # Bill / payment aggregates
    bill_cols = [f"bill_amt{i}" for i in range(1, 7) if f"bill_amt{i}" in df.columns]
    pay_amt_cols = [f"pay_amt{i}" for i in range(1, 7) if f"pay_amt{i}" in df.columns]

    if len(bill_cols) > 0:
        df["bill_amt_mean"] = df[bill_cols].mean(axis=1)
        df["bill_amt_std"] = df[bill_cols].std(axis=1).fillna(0)
        df["bill_amt_max"] = df[bill_cols].max(axis=1)
        df["bill_trend_1_vs_6"] = df[bill_cols[0]] - df[bill_cols[-1]]
        created += ["bill_amt_mean", "bill_amt_std", "bill_amt_max", "bill_trend_1_vs_6"]

    if len(pay_amt_cols) > 0:
        df["pay_amt_mean"] = df[pay_amt_cols].mean(axis=1)
        df["pay_amt_std"] = df[pay_amt_cols].std(axis=1).fillna(0)
        df["pay_amt_sum"] = df[pay_amt_cols].sum(axis=1)
        created += ["pay_amt_mean", "pay_amt_std", "pay_amt_sum"]

    if len(bill_cols) > 0 and len(pay_amt_cols) > 0:
        df["pay_to_bill_total_ratio"] = (
            df[pay_amt_cols].sum(axis=1) / (df[bill_cols].abs().sum(axis=1) + 1)
        )
        created.append("pay_to_bill_total_ratio")

    # Compact summaries
    util_cols = [c for c in df.columns if c.startswith("util_ratio")]
    ptr_cols = [c for c in df.columns if c.startswith("pay_to_bill_ratio")]
    if len(util_cols) > 0:
        df["util_ratio_mean"] = df[util_cols].mean(axis=1)
        df["util_ratio_max"] = df[util_cols].max(axis=1)
        created += ["util_ratio_mean", "util_ratio_max"]
    if len(ptr_cols) > 0:
        df["pay_to_bill_ratio_mean"] = df[ptr_cols].mean(axis=1)
        created.append("pay_to_bill_ratio_mean")

    return df, sorted(list(set(created)))


## 1) Data loading + quality checks

In [ ]:
df = load_credit_dataset(DATA_PATH)
target_col = detect_target_column(list(df.columns))

# Drop ID-like columns
id_cols = [c for c in df.columns if c in {"id", "id_"} or c.startswith("id_")]
if id_cols:
    df = df.drop(columns=id_cols)

# Feature engineering
df, created_features = add_feature_engineering(df)

print("Shape:", df.shape)
print("Target:", target_col)
print("Created engineered features:", len(created_features))
print("Sample engineered features:", created_features[:12])

display(df.head())

print("\nTarget counts:")
display(df[target_col].value_counts(dropna=False))

print("\nTarget ratios:")
display(df[target_col].value_counts(normalize=True).rename("ratio"))

print("\nMissing values summary:")
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
if len(missing_summary) == 0:
    print("No missing values detected.")
else:
    display(missing_summary)

print("\nData types summary:")
display(df.dtypes.value_counts())

print("\nDescriptive statistics (numeric):")
display(df.describe(include=[np.number]).T.head(30))


## 2) EDA - class imbalance and key distributions

In [ ]:
# Target imbalance visualization
plt.figure(figsize=(5, 4))
ax = sns.countplot(x=df[target_col])
plt.title("Target distribution (0=non-default, 1=default)")
for p in ax.patches:
    count = int(p.get_height())
    ratio = count / len(df)
    ax.annotate(f"{count} ({ratio:.1%})", (p.get_x() + p.get_width()/2, p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.show()

# Distributions for key variables
key_cols = [
    c for c in [
        "limit_bal", "age", "bill_amt1", "bill_amt_mean", "pay_amt1",
        "pay_amt_mean", "util_ratio_mean", "delinquency_count", "delinquency_max"
    ] if c in df.columns
]

if len(key_cols) > 0:
    fig, axes = plt.subplots(len(key_cols), 2, figsize=(12, 3 * len(key_cols)))
    if len(key_cols) == 1:
        axes = np.array([axes])

    for i, col in enumerate(key_cols):
        sns.histplot(df[col], bins=40, ax=axes[i, 0], kde=False)
        axes[i, 0].set_title(f"Distribution - {col}")

        sns.boxplot(data=df, x=target_col, y=col, ax=axes[i, 1])
        axes[i, 1].set_title(f"{col} by target")

    plt.tight_layout()
    plt.show()


## 3) EDA - default vs non-default comparison table

In [ ]:
compare_cols = [
    c for c in [
        "limit_bal", "age", "bill_amt_mean", "pay_amt_mean", "util_ratio_mean",
        "pay_to_bill_total_ratio", "delinquency_count", "delinquency_max", "delinquency_mean"
    ] if c in df.columns
]

comparison_rows = []
for col in compare_cols:
    mean_non_default = df.loc[df[target_col] == 0, col].mean()
    mean_default = df.loc[df[target_col] == 1, col].mean()
    median_non_default = df.loc[df[target_col] == 0, col].median()
    median_default = df.loc[df[target_col] == 1, col].median()

    pct_diff = ((mean_default - mean_non_default) / (abs(mean_non_default) + 1e-9)) * 100

    comparison_rows.append({
        "feature": col,
        "mean_non_default": mean_non_default,
        "mean_default": mean_default,
        "median_non_default": median_non_default,
        "median_default": median_default,
        "pct_diff_default_vs_non_default": pct_diff,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    "pct_diff_default_vs_non_default",
    key=lambda s: s.abs(),
    ascending=False,
)

display(comparison_df.head(15))
comparison_df.to_csv(OUTPUT_DIR / "eda_default_vs_non_default.csv", index=False)
print("Saved:", OUTPUT_DIR / "eda_default_vs_non_default.csv")


## 4) EDA - correlations

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).copy()

if target_col in numeric_df.columns:
    corr_target = (
        numeric_df.corr(numeric_only=True)[target_col]
        .drop(target_col)
        .sort_values(key=lambda s: s.abs(), ascending=False)
    )

    print("Top correlations with target (absolute):")
    display(corr_target.head(20).to_frame("corr_with_target"))

    top_cols = corr_target.head(12).index.tolist() + [target_col]
    plt.figure(figsize=(10, 6))
    sns.heatmap(numeric_df[top_cols].corr(numeric_only=True), cmap="coolwarm", center=0)
    plt.title("Correlation heatmap (top features + target)")
    plt.tight_layout()
    plt.show()


## 5) Train/test split + class imbalance setup

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nClass distribution in train:")
display(y_train.value_counts())
display(y_train.value_counts(normalize=True).rename("ratio"))

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / max(pos, 1)
print(f"scale_pos_weight (neg/pos): {scale_pos_weight:.2f}")


In [ ]:
# SMOTE for MLP final training
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE (for MLP train):")
display(pd.Series(y_train_smote).value_counts())


## 6) Define required models

In [ ]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    C=1.0,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
    "Neural Network (MLP)": Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(64, 32),
                    alpha=1e-4,
                    max_iter=300,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

print("Models loaded:")
for k in models:
    print("-", k)


## 7) Cross-validation comparison (quick mode)


In [ ]:
# Quick CV to keep notebook responsive
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Sample train set for faster CV (deterministic)
if len(X_train) > CV_SAMPLE_MAX:
    X_cv = X_train.sample(n=CV_SAMPLE_MAX, random_state=RANDOM_STATE)
    y_cv = y_train.loc[X_cv.index]
else:
    X_cv = X_train.copy()
    y_cv = y_train.copy()

print(f"CV folds: {CV_FOLDS}")
print(f"CV sample size: {len(X_cv)} / {len(X_train)}")

cv_models = {
    "Logistic Regression": clone(models["Logistic Regression"]),
    "Random Forest": clone(models["Random Forest"]),
    # Lighter XGBoost only for CV speed (final training still uses full model config)
    "XGBoost": XGBClassifier(
        n_estimators=120,
        max_depth=3,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
}

if CV_INCLUDE_MLP:
    cv_models["Neural Network (MLP)"] = ImbPipeline([
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf", clone(models["Neural Network (MLP)"])),
    ])

cv_rows = []
for model_name, model in cv_models.items():
    scores = cross_validate(
        model,
        X_cv,
        y_cv,
        cv=cv,
        scoring={
            "auc": "roc_auc",
            "f1": "f1",
            "precision": "precision",
            "recall": "recall",
        },
        n_jobs=-1,
        error_score="raise",
    )

    cv_rows.append({
        "model": model_name,
        "cv_auc_mean": round(float(np.mean(scores["test_auc"])), 4),
        "cv_auc_std": round(float(np.std(scores["test_auc"])), 4),
        "cv_f1_mean": round(float(np.mean(scores["test_f1"])), 4),
        "cv_f1_std": round(float(np.std(scores["test_f1"])), 4),
        "cv_precision_mean": round(float(np.mean(scores["test_precision"])), 4),
        "cv_recall_mean": round(float(np.mean(scores["test_recall"])), 4),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("cv_auc_mean", ascending=False)
display(cv_df)
cv_df.to_csv(OUTPUT_DIR / "cv_results.csv", index=False)
print("Saved:", OUTPUT_DIR / "cv_results.csv")


## 8) Final training + test evaluation

In [ ]:
results = []
report_rows = []
model_test_probs = {}
fitted_models = {}

plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")

for model_name, model in models.items():
    if model_name == "Neural Network (MLP)":
        model.fit(X_train_smote, y_train_smote)
    else:
        model.fit(X_train, y_train)

    fitted_models[model_name] = model

    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= DEFAULT_THRESHOLD).astype(int)
    model_test_probs[model_name] = y_prob

    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results.append({
        "model": model_name,
        "auc": round(float(auc), 4),
        "f1_score": round(float(f1), 4),
        "precision": round(float(prec), 4),
        "recall": round(float(rec), 4),
        "tn": int(cm[0, 0]),
        "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]),
        "tp": int(cm[1, 1]),
    })

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    for label, vals in report.items():
        if isinstance(vals, dict):
            report_rows.append({
                "model": model_name,
                "label": label,
                "precision": vals.get("precision", np.nan),
                "recall": vals.get("recall", np.nan),
                "f1_score": vals.get("f1-score", np.nan),
                "support": vals.get("support", np.nan),
            })

    # confusion matrix figure
    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix - {model_name} (thr={DEFAULT_THRESHOLD})")
    plt.tight_layout()
    safe_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    fig.savefig(CM_DIR / f"cm_{safe_name}.png")
    plt.close(fig)

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc:.3f})")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - Test set")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png")
plt.show()

results_df = pd.DataFrame(results).sort_values("auc", ascending=False)
report_df = pd.DataFrame(report_rows)

final_df = results_df.merge(cv_df, on="model", how="left")

preferred_cols = [
    "model",
    "cv_auc_mean", "cv_auc_std", "cv_f1_mean", "cv_f1_std",
    "cv_precision_mean", "cv_recall_mean",
    "auc", "f1_score", "precision", "recall",
    "tn", "fp", "fn", "tp",
]
final_df = final_df[[c for c in preferred_cols if c in final_df.columns]]

display(final_df)

final_df.to_csv(OUTPUT_DIR / "model_results.csv", index=False)
report_df.to_csv(OUTPUT_DIR / "classification_report_full.csv", index=False)

print("Saved:", OUTPUT_DIR / "model_results.csv")
print("Saved:", OUTPUT_DIR / "classification_report_full.csv")


## 9) Threshold decision analysis (business trade-off)

In [ ]:
# Select best model by AUC on test for threshold analysis
best_model_name = final_df.sort_values("auc", ascending=False).iloc[0]["model"]
best_probs = model_test_probs[best_model_name]

threshold_rows = []
for t in np.arange(0.25, 0.76, 0.05):
    y_pred_t = (best_probs >= t).astype(int)
    cm_t = confusion_matrix(y_test, y_pred_t)

    threshold_rows.append({
        "model": best_model_name,
        "threshold": round(float(t), 2),
        "auc": round(float(roc_auc_score(y_test, best_probs)), 4),  # independent from threshold
        "f1": round(float(f1_score(y_test, y_pred_t)), 4),
        "precision": round(float(precision_score(y_test, y_pred_t, zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, y_pred_t, zero_division=0)), 4),
        "tn": int(cm_t[0, 0]),
        "fp": int(cm_t[0, 1]),
        "fn": int(cm_t[1, 0]),
        "tp": int(cm_t[1, 1]),
    })

threshold_df = pd.DataFrame(threshold_rows)

# Recommendation rule: prioritize detection while keeping minimum precision
candidates = threshold_df[threshold_df["precision"] >= 0.35]
if len(candidates) > 0:
    recommended = candidates.sort_values(["f1", "recall"], ascending=False).iloc[0]
else:
    recommended = threshold_df.sort_values(["f1", "recall"], ascending=False).iloc[0]

recommended_threshold = float(recommended["threshold"])

print(f"Best model for threshold analysis: {best_model_name}")
print(f"Recommended threshold: {recommended_threshold:.2f}")
print("Reason: maximize F1 while controlling precision.")

display(threshold_df.sort_values("f1", ascending=False).head(12))

threshold_df.to_csv(OUTPUT_DIR / "threshold_analysis_best_model.csv", index=False)
print("Saved:", OUTPUT_DIR / "threshold_analysis_best_model.csv")

# threshold curves
plt.figure(figsize=(8, 5))
plt.plot(threshold_df["threshold"], threshold_df["f1"], label="F1")
plt.plot(threshold_df["threshold"], threshold_df["precision"], label="Precision")
plt.plot(threshold_df["threshold"], threshold_df["recall"], label="Recall")
plt.axvline(recommended_threshold, color="red", linestyle="--", label=f"Recommended={recommended_threshold:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Metric")
plt.title(f"Threshold trade-off - {best_model_name}")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "threshold_tradeoff.png")
plt.show()


## 10) Interpretation - feature importance + economic meaning

In [ ]:
importance_rows = []

# Random Forest importances
rf_model = fitted_models.get("Random Forest")
if rf_model is not None and hasattr(rf_model, "feature_importances_"):
    for f, v in zip(X_train.columns, rf_model.feature_importances_):
        importance_rows.append({"model": "Random Forest", "feature": f, "importance": float(v)})

# XGBoost importances
xgb_model = fitted_models.get("XGBoost")
if xgb_model is not None and hasattr(xgb_model, "feature_importances_"):
    for f, v in zip(X_train.columns, xgb_model.feature_importances_):
        importance_rows.append({"model": "XGBoost", "feature": f, "importance": float(v)})

importance_df = pd.DataFrame(importance_rows)
if len(importance_df) > 0:
    importance_df.to_csv(OUTPUT_DIR / "feature_importance_summary.csv", index=False)

# Logistic coefficients (signed)
log_model = fitted_models["Logistic Regression"].named_steps["model"]
coef_series = pd.Series(log_model.coef_[0], index=X_train.columns)

log_pos = coef_series.sort_values(ascending=False).head(10).to_frame("coef")
log_neg = coef_series.sort_values(ascending=True).head(10).to_frame("coef")

print("Top positive Logistic coefficients (higher default probability)")
display(log_pos)

print("Top negative Logistic coefficients (lower default probability)")
display(log_neg)

for m in importance_df["model"].unique() if len(importance_df) > 0 else []:
    print(f"\nTop features - {m}")
    display(
        importance_df[importance_df["model"] == m]
        .sort_values("importance", ascending=False)
        .head(10)
    )


print('Saved:', OUTPUT_DIR / 'feature_importance_summary.csv')


## 11) Business observations (finance-oriented, jury-ready)


In [ ]:
business_notes = []

# 1) Class imbalance note
default_rate = float(df[target_col].mean())
business_notes.append(
    f"Default rate is {default_rate:.1%}. This confirms an imbalanced risk problem and justifies class-weighted models and SMOTE."
)

# 2) Behavior difference note
if 'comparison_df' in globals() and len(comparison_df) > 0:
    top3 = comparison_df.head(3)['feature'].tolist()
    business_notes.append(
        "Defaulting clients show a weaker repayment profile: lower payment intensity versus billed amount and more signs of delinquency."
    )
    business_notes.append(
        f"In our data, the most discriminative behavioral indicators include: {', '.join(top3)}."
    )

# 3) Model performance note
best_row = final_df.sort_values('auc', ascending=False).iloc[0]
business_notes.append(
    f"Best model by AUC is {best_row['model']} (AUC={best_row['auc']:.3f}, F1={best_row['f1_score']:.3f}, Recall={best_row['recall']:.3f})."
)

# 4) Threshold/business trade-off note
business_notes.append(
    f"Threshold is not fixed at 0.50: we selected {recommended_threshold:.2f} by balancing recall and precision to reduce undetected defaults while controlling false alarms."
)

# 5) Feature engineering defense note
feature_defense = {
    'util_ratio': 'High credit utilization indicates liquidity pressure and potential repayment stress.',
    'pay_to_bill_ratio': 'A low payment-to-bill ratio suggests inability to cover outstanding obligations.',
    'delinquency': 'Past late-payment behavior is a direct signal of future credit risk persistence.',
    'bill_trend_1_vs_6': 'A worsening bill trend can indicate rising leverage and deteriorating financial discipline.'
}

# 6) Tuning rationale note
business_notes.append(
    "We used small manual tuning to keep models explainable, robust, and less prone to overfitting in a student-project setting."
)

print('Business insights for oral presentation:')
for i, note in enumerate(business_notes, start=1):
    print(f"{i}. {note}")

print("\nFeature engineering rationale (Q&A ready):")
for k, v in feature_defense.items():
    print(f"- {k}: {v}")

pd.DataFrame({'business_observation': business_notes}).to_csv(
    OUTPUT_DIR / 'business_observations_for_slides.csv',
    index=False,
)
pd.DataFrame(
    [{'feature_family': k, 'why_it_matters': v} for k, v in feature_defense.items()]
).to_csv(
    OUTPUT_DIR / 'feature_engineering_defense_notes.csv',
    index=False,
)
print('Saved:', OUTPUT_DIR / 'business_observations_for_slides.csv')
print('Saved:', OUTPUT_DIR / 'feature_engineering_defense_notes.csv')


## Files for final presentation
- `outputs/model_results.csv`: core comparison table (CV + test metrics)
- `outputs/cv_results.csv`: model stability (3-fold)
- `outputs/classification_report_full.csv`: precision/recall/f1 by class
- `outputs/threshold_analysis_best_model.csv`: threshold trade-off table
- `outputs/threshold_tradeoff.png`: precision/recall/f1 vs threshold figure
- `outputs/feature_importance_summary.csv`: RF/XGB importances
- `outputs/business_observations_for_slides.csv`: finance-style oral points
- `outputs/feature_engineering_defense_notes.csv`: short answers for why these features?
- `outputs/roc_curves.png`
- `outputs/confusion_matrices/*.png`


## 12) Presentation mode (lean, jury-friendly)
Use this section to avoid showing too much detail during the oral.


In [ ]:

# Build a compact pack of only the most useful presentation elements

kpi_rows = []

# Global KPI
kpi_rows.append({'item': 'default_rate', 'value': round(float(df[target_col].mean()), 4)})

# Best model KPI
best = final_df.sort_values('auc', ascending=False).iloc[0]
for col in ['model','auc','f1_score','precision','recall']:
    if col in best.index:
        kpi_rows.append({'item': f'best_{col}', 'value': best[col]})

# Threshold KPI
kpi_rows.append({'item': 'recommended_threshold', 'value': round(float(recommended_threshold), 2)})

presentation_kpi_df = pd.DataFrame(kpi_rows)
display(presentation_kpi_df)

# Top 5 risk drivers (simple source)
if 'importance_df' in globals() and len(importance_df) > 0:
    best_model_name = str(best['model']) if 'model' in best.index else 'XGBoost'
    candidate = importance_df[importance_df['model'].str.contains(best_model_name.split()[0], case=False, na=False)]
    if len(candidate) == 0:
        candidate = importance_df
    top5_drivers = candidate.sort_values('importance', ascending=False).head(5)[['feature','importance']]
else:
    top5_drivers = pd.DataFrame(columns=['feature','importance'])

print('Top 5 risk drivers (for slides):')
display(top5_drivers)

# Compact outputs for presentation
presentation_kpi_df.to_csv(OUTPUT_DIR / 'presentation_kpis.csv', index=False)
top5_drivers.to_csv(OUTPUT_DIR / 'presentation_top5_drivers.csv', index=False)

print('Saved:', OUTPUT_DIR / 'presentation_kpis.csv')
print('Saved:', OUTPUT_DIR / 'presentation_top5_drivers.csv')
print('Use these 2 files + roc_curves.png + one confusion matrix in oral presentation.')
